# Advanced Experimentation Walkthrough

This notebook reproduces the primary onboarding experiment readout, adds an 80%-power MDE calculation, and demonstrates a CUPED-style pre-treatment covariate adjustment.

**Methodological note:** because this is a new-user onboarding experiment, there is no genuine pre-period activation outcome. The variance-reduction example therefore uses a treatment-blind propensity score built only from acquisition channel and device. The unadjusted primary analysis remains confirmatory.

In [ ]:
from src.generate_dataset import generate_users
from src.experiment import two_proportion_test
from src.power import minimum_detectable_effect, two_sided_power
from src.cuped import cuped_adjust_activation


In [ ]:
rows = generate_users(12_000, seed=20_260_808)
primary = two_proportion_test(rows, 'activated_7d')
n_control = sum(row['variant'] == 'control' for row in rows)
n_treatment = sum(row['variant'] == 'treatment' for row in rows)
print(f'control: {n_control:,} users @ {primary.control_rate:.2%}')
print(f'treatment: {n_treatment:,} users @ {primary.treatment_rate:.2%}')
print(f'raw lift: {primary.absolute_lift * 100:+.2f} pp; p={primary.p_value:.4f}')


In [ ]:
mde = minimum_detectable_effect(primary.control_rate, n_control, n_treatment, target_power=0.80)
observed_power = two_sided_power(primary.control_rate, primary.absolute_lift, n_control, n_treatment)
print(f'80% power MDE: {mde * 100:.2f} pp')
print(f'planning power at observed lift: {observed_power:.1%}')
print('Interpretation: the observed lift is smaller than the 80%-power MDE, so this significant result is a favorable realization rather than an effect size the design would detect 80% of the time.')


In [ ]:
cuped = cuped_adjust_activation(rows)
print(f'theta: {cuped.theta:.4f}')
print(f'covariate variance reduction: {cuped.variance_reduction:.2%}')
print(f'raw lift: {cuped.raw_difference * 100:+.2f} pp')
print(f'adjusted lift: {cuped.adjusted_difference * 100:+.2f} pp')
print(f'adjusted p-value: {cuped.p_value:.4f}')


## Decision discipline

- The original unadjusted 7-day activation analysis remains the primary decision statistic because it was defined in the experiment contract.
- Power/MDE is a planning lens, not a way to reinterpret significance after seeing the result.
- The CUPED-style analysis is a variance-reduction demonstration and sensitivity check, not a post-hoc replacement for the pre-specified primary analysis.
- In a production experiment, the covariate definition and adjustment method should be frozen before reading treatment outcomes.